# Cardiotoxicity Prediction Using Molecular Fingerprints and Machine Learning

# Project Objective
Cardiotoxicity is a major cause of late-stage drug attrition and post-market withdrawal. This project develops machine learning models capable of predicting cardiotoxic compounds directly from molecular structure using molecular fingerprints and physicochemical descriptors.

The workflow emphasizes methodological rigor through:
- Structural deduplication using canonical SMILES
- Scaffold-based train-test partitioning
- Leakage-free threshold optimization
- Evaluation on unseen molecular scaffolds






The PubChem BioAssay AID 588834 dataset was used. Only molecular structures and binary activity labels were retained. Inconclusive records and duplicate molecular structures were removed prior to model development.

In [31]:
import pandas as pd
df_raw = pd.read_csv("AID_588834_datatable.csv")

df_raw = df_raw[
    df_raw["PUBCHEM_ACTIVITY_OUTCOME"].isin(
        ["Active", "Inactive", "Inconclusive"]
    )
].copy()

print(df_raw.shape)

(5381, 43)


In [4]:
df = df_raw[
    [
        "PUBCHEM_EXT_DATASOURCE_SMILES",
        "PUBCHEM_ACTIVITY_OUTCOME"
    ]
].copy()

df.columns = ["SMILES", "Activity"]

df = df[df["SMILES"].notnull()]

df = df[
    df["Activity"].isin(
        ["Active", "Inactive"]
    )
].reset_index(drop=True)

df["label"] = df["Activity"].map(
    {
        "Inactive": 0,
        "Active": 1
    }
)

print(df.shape)
print(df["label"].value_counts())

(4754, 3)
label
0    4092
1     662
Name: count, dtype: int64


## Molecular Standardization

Molecular structures were parsed using RDKit. Counterions and salts were removed where possible and compounds were standardized using canonical SMILES representations.

This process enables reliable duplicate detection and ensures consistent molecular representations.

In [5]:
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

df["mol"] = df["SMILES"].apply(
    lambda x: Chem.MolFromSmiles(str(x))
)

df = df[df["mol"].notnull()].reset_index(drop=True)

remover = SaltRemover()

df["mol"] = df["mol"].apply(
    lambda m: remover.StripMol(m)
)

print(df.shape)

[09:08:43] WARNING: not removing hydrogen atom without neighbors
[09:08:43] WARNING: not removing hydrogen atom without neighbors


(4754, 4)


In [6]:
df["canonical_smiles"] = df["mol"].apply(
    lambda m: Chem.MolToSmiles(
        m,
        canonical=True
    )
)

duplicates = df.duplicated(
    subset=["canonical_smiles"]
).sum()

print("Duplicates:", duplicates)

df = (
    df.drop_duplicates(
        subset=["canonical_smiles"]
    )
    .reset_index(drop=True)
)

df["mol"] = df["canonical_smiles"].apply(
    Chem.MolFromSmiles
)

print("Remaining:", len(df))
print(df["label"].value_counts())

Duplicates: 621
Remaining: 4133
label
0    3578
1     555
Name: count, dtype: int64


[09:09:05] WARNING: not removing hydrogen atom without neighbors
[09:09:05] WARNING: not removing hydrogen atom without neighbors


In [8]:
import numpy as np
from rdkit import DataStructs
from rdkit.Chem import AllChem

morgan_gen = AllChem.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

morgan_features = []

for mol in df["mol"]:

    fp = morgan_gen.GetFingerprint(mol)

    arr = np.zeros(
        (2048,),
        dtype=np.int8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    morgan_features.append(arr)

morgan_features = np.array(morgan_features)

print(morgan_features.shape)
print(morgan_features.dtype)

(4133, 2048)
int8


## Molecular Feature Engineering

Three complementary feature representations were generated:

1. Morgan fingerprints (ECFP4, 2048 bits)
2. MACCS structural keys
3. Physicochemical descriptors

These features capture structural fragments, predefined chemical motifs, and global molecular properties.

In [9]:
from rdkit.Chem import MACCSkeys

maccs_features = []

for mol in df["mol"]:

    fp = MACCSkeys.GenMACCSKeys(mol)

    arr = np.zeros(
        (len(fp),),
        dtype=np.int8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    maccs_features.append(arr)

maccs_features = np.array(maccs_features)

print(maccs_features.shape)

(4133, 167)


In [10]:
from rdkit.Chem import Descriptors

descriptor_features = []

for mol in df["mol"]:

    descriptor_features.append([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol)
    ])

descriptor_features = np.array(
    descriptor_features
)

print(descriptor_features.shape)

(4133, 7)


In [11]:
X = np.hstack([
    morgan_features,
    maccs_features,
    descriptor_features
])

y = df["label"].values

print(X.shape)
print(y.shape)

(4133, 2222)
(4133,)


## Scaffold-Based Train/Test Split

To evaluate generalization to unseen chemical scaffolds, compounds were partitioned using Bemis–Murcko scaffolds. Entire scaffold families were assigned exclusively to either the training or test set, preventing structural leakage.

In [12]:
from rdkit.Chem.Scaffolds import MurckoScaffold

def get_scaffold(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return ""

    return MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol
    )

df["scaffold"] = df["canonical_smiles"].apply(
    get_scaffold
)

print(
    "Unique scaffolds:",
    df["scaffold"].nunique()
)

[09:12:19] WARNING: not removing hydrogen atom without neighbors
[09:12:19] WARNING: not removing hydrogen atom without neighbors


Unique scaffolds: 1584


In [13]:
scaffold_to_indices = df.groupby(
    "scaffold"
).indices

train_idx = []
test_idx = []

target_train_size = int(
    0.80 * len(df)
)

current_train_size = 0

sorted_scaffolds = sorted(
    scaffold_to_indices.items(),
    key=lambda item: len(item[1]),
    reverse=True
)

for scaffold, indices in sorted_scaffolds:

    if current_train_size < target_train_size:

        train_idx.extend(indices)

        current_train_size += len(indices)

    else:

        test_idx.extend(indices)

print("Train compounds:", len(train_idx))
print("Test compounds :", len(test_idx))

Train compounds: 3306
Test compounds : 827


In [14]:
X_train_raw = X[train_idx]
X_test_raw = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

print(X_train_raw.shape)
print(X_test_raw.shape)

print("\nTrain Labels")
print(pd.Series(y_train).value_counts())

print("\nTest Labels")
print(pd.Series(y_test).value_counts())

(3306, 2222)
(827, 2222)

Train Labels
0    2951
1     355
Name: count, dtype: int64

Test Labels
0    627
1    200
Name: count, dtype: int64


## Feature Selection and Scaling

A two-stage feature reduction strategy was applied.

1. Near-constant features were removed using variance thresholding.
2. Informative variables were selected using mutual information.

Feature scaling was subsequently applied to normalize numerical ranges prior to model training.

In [15]:
from sklearn.feature_selection import VarianceThreshold

variance_selector = VarianceThreshold(
    threshold=0.01
)

X_train_fs = variance_selector.fit_transform(
    X_train_raw
)

X_test_fs = variance_selector.transform(
    X_test_raw
)

print(X_train_fs.shape)
print(X_test_fs.shape)

(3306, 778)
(827, 778)


In [16]:
from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_classif
)

k_features = min(
    500,
    X_train_fs.shape[1]
)

mi_selector = SelectKBest(
    mutual_info_classif,
    k=k_features
)

X_train_fs = mi_selector.fit_transform(
    X_train_fs,
    y_train
)

X_test_fs = mi_selector.transform(
    X_test_fs
)

print(X_train_fs.shape)
print(X_test_fs.shape)

(3306, 500)
(827, 500)


In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_final = scaler.fit_transform(
    X_train_fs
)

X_test_final = scaler.transform(
    X_test_fs
)

print(X_train_final.shape)
print(X_test_final.shape)

(3306, 500)
(827, 500)


## Leakage-Free Threshold Optimization

Decision thresholds were optimized using out-of-fold predictions obtained through five-fold cross-validation.

Feature selection and scaling were performed independently within each fold to prevent information leakage from validation data into model development.

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def optimize_threshold_internal(
    model_type,
    X_tr_raw,
    y_tr
):

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    oof_probs = np.zeros(
        len(y_tr)
    )

    for tr_idx, val_idx in skf.split(
        X_tr_raw,
        y_tr
    ):

        X_fold_t_raw = X_tr_raw[tr_idx]
        y_fold_t = y_tr[tr_idx]

        X_fold_v_raw = X_tr_raw[val_idx]

        # variance

        v_selector = VarianceThreshold(
            threshold=0.01
        )

        X_t_f = v_selector.fit_transform(
            X_fold_t_raw
        )

        X_v_f = v_selector.transform(
            X_fold_v_raw
        )

        # MI

        k_fold = min(
            500,
            X_t_f.shape[1]
        )

        mi_selector = SelectKBest(
            mutual_info_classif,
            k=k_fold
        )

        X_t_f = mi_selector.fit_transform(
            X_t_f,
            y_fold_t
        )

        X_v_f = mi_selector.transform(
            X_v_f
        )

        # scale

        scaler_f = StandardScaler()

        X_t_f = scaler_f.fit_transform(
            X_t_f
        )

        X_v_f = scaler_f.transform(
            X_v_f
        )

        # model

        if model_type == "rf":

            model = RandomForestClassifier(
                n_estimators=500,
                max_features="sqrt",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )

        else:

            ratio = (
                (len(y_fold_t) - np.sum(y_fold_t))
                /
                np.sum(y_fold_t)
            )

            model = xgb.XGBClassifier(
                n_estimators=500,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=1,
                reg_lambda=3,
                scale_pos_weight=ratio,
                random_state=42,
                n_jobs=-1
            )

        model.fit(
            X_t_f,
            y_fold_t
        )

        oof_probs[val_idx] = (
            model.predict_proba(
                X_v_f
            )[:,1]
        )

    thresholds = np.arange(
        0.10,
        0.91,
        0.05
    )

    best_t = 0.50
    best_mcc = -1

    for t in thresholds:

        preds = (
            oof_probs >= t
        ).astype(int)

        mcc = matthews_corrcoef(
            y_tr,
            preds
        )

        if mcc > best_mcc:

            best_mcc = mcc
            best_t = t

    return best_t

## Random Forest Model

A Random Forest classifier was trained using class-balanced learning. The decision threshold was optimized using out-of-fold predictions obtained from five-fold cross-validation.

In [20]:

from sklearn.ensemble import RandomForestClassifier
print("Optimizing RF threshold...")

rf_threshold = optimize_threshold_internal(
    "rf",
    X_train_raw,
    y_train
)

print("Best RF Threshold:", rf_threshold)

Optimizing RF threshold...
Best RF Threshold: 0.45000000000000007


In [21]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_final,
    y_train
)

RandomForestClassifier(class_weight='balanced', n_estimators=500, n_jobs=-1,
                       random_state=42)

In [22]:
rf_probs = rf_model.predict_proba(
    X_test_final
)[:,1]

rf_preds = (
    rf_probs >= rf_threshold
).astype(int)

In [23]:
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    matthews_corrcoef
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        rf_probs
    )
)

print(
    "MCC:",
    matthews_corrcoef(
        y_test,
        rf_preds
    )
)

print(
    classification_report(
        y_test,
        rf_preds
    )
)

ROC-AUC: 0.8970215311004786
MCC: 0.6254223866229867
              precision    recall  f1-score   support

           0       0.89      0.95      0.92       627
           1       0.80      0.61      0.70       200

    accuracy                           0.87       827
   macro avg       0.84      0.78      0.81       827
weighted avg       0.87      0.87      0.86       827



## Split Validation

The scaffold partition was verified by confirming that no scaffold families were shared between the training and test sets.

In [25]:
print("Unique scaffolds in train:",
      len(set(df.iloc[train_idx]["scaffold"])))

print("Unique scaffolds in test:",
      len(set(df.iloc[test_idx]["scaffold"])))

print(
    "Shared scaffolds:",
    len(
        set(df.iloc[train_idx]["scaffold"])
        &
        set(df.iloc[test_idx]["scaffold"])
    )
)

Unique scaffolds in train: 757
Unique scaffolds in test: 827
Shared scaffolds: 0


## XGBoost Benchmark

An XGBoost classifier was trained using the same feature set and scaffold split to provide a comparative benchmark against the Random Forest model.

In [27]:

import xgboost as xgb
print("Optimizing XGB threshold...")

xgb_threshold = optimize_threshold_internal(
    "xgb",
    X_train_raw,
    y_train
)

print("Best XGB Threshold:", xgb_threshold)

Optimizing XGB threshold...
Best XGB Threshold: 0.7500000000000002


In [28]:
ratio = (len(y_train) - np.sum(y_train)) / np.sum(y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=3,
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_final, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [29]:
xgb_probs = xgb_model.predict_proba(X_test_final)[:,1]

xgb_preds = (
    xgb_probs >= xgb_threshold
).astype(int)

In [30]:
print(
    "ROC-AUC:",
    roc_auc_score(y_test, xgb_probs)
)

print(
    "MCC:",
    matthews_corrcoef(y_test, xgb_preds)
)

print(
    classification_report(
        y_test,
        xgb_preds
    )
)

ROC-AUC: 0.8912360446570973
MCC: 0.6020738427175252
              precision    recall  f1-score   support

           0       0.90      0.92      0.91       627
           1       0.72      0.67      0.69       200

    accuracy                           0.86       827
   macro avg       0.81      0.79      0.80       827
weighted avg       0.85      0.86      0.86       827



## Results and Discussion

Random Forest achieved the strongest overall performance:

| Model | ROC-AUC | MCC |
|---------|---------|---------|
| Random Forest | 0.897 | 0.625 |
| XGBoost | 0.891 | 0.602 |

The use of scaffold-based splitting provides a more realistic estimate of model performance on structurally novel compounds compared with conventional random train-test splits.

